# Train YOLOv11s on Kaggle P100 (with auto-save callback)

**Khác với Colab notebook:**
- Auto-save best.pt mỗi epoch về Kaggle dataset output (không bao giờ mất)
- Reduce epochs 100 → 70 (đã thấy plateau từ epoch 75 trên Colab)
- Better resume nếu disconnect

## Setup Kaggle

1. New Notebook → Settings → Accelerator: **GPU P100** (bắt buộc)
2. Add Data: upload `yolo_dataset_merged.zip` (~547MB) — sẽ ở `/kaggle/input/`
3. Save Version → Save & Run All (khi muốn run từ đầu)

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install ultralytics

In [ ]:
!pip install -q ultralytics 'numpy<2.1'
import ultralytics
ultralytics.checks()

## 3. Extract dataset từ Kaggle Input

In [ ]:
import os, zipfile

# Tìm zip trong /kaggle/input/
import glob
zip_files = glob.glob('/kaggle/input/**/*.zip', recursive=True)
print('Found zips:', zip_files)

ZIP_PATH = zip_files[0]  # First zip
EXTRACT_TO = '/kaggle/working/dataset'

if not os.path.exists(EXTRACT_TO):
    os.makedirs(EXTRACT_TO)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_TO)

for root, dirs, files in os.walk(EXTRACT_TO):
    if 'data.yaml' in files:
        DATA_YAML = os.path.join(root, 'data.yaml')
        break
print(f'data.yaml: {DATA_YAML}')

## 4. Fix paths

In [ ]:
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

DATASET_ROOT = os.path.dirname(DATA_YAML)
cfg['train'] = os.path.join(DATASET_ROOT, 'train', 'images')
cfg['val']   = os.path.join(DATASET_ROOT, 'valid', 'images')
cfg['test']  = os.path.join(DATASET_ROOT, 'test', 'images')

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"Classes ({cfg['nc']}): {cfg['names']}")
print(f"Train: {cfg['train']}")
print(f"Val:   {cfg['val']}")

## 5. Auto-save callback — bảo hiểm chống mất train

Sau MỖI epoch, copy best.pt sang `/kaggle/working/` (sẽ tự động commit khi save notebook version).

In [ ]:
import shutil
from ultralytics import YOLO
from ultralytics.utils import callbacks

BACKUP_DIR = '/kaggle/working/model_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)

def on_fit_epoch_end(trainer):
    """Sau mỗi epoch, copy best.pt + last.pt sang BACKUP_DIR."""
    weights_dir = trainer.save_dir / 'weights'
    for fname in ['best.pt', 'last.pt']:
        src = weights_dir / fname
        if src.exists():
            shutil.copy(src, f'{BACKUP_DIR}/{fname}')
    epoch = trainer.epoch + 1
    metrics = trainer.metrics if trainer.metrics else {}
    print(f'[BACKUP] Epoch {epoch} → {BACKUP_DIR}/best.pt | mAP50={metrics.get("metrics/mAP50(B)", 0):.4f}')

print('Callback registered. Ready to train.')

## 6. Train YOLOv11s — 70 epochs (đã learn từ lần trước)

Dựa trên Colab run trước:
- mAP@50 epoch 80 = 0.582 (plateau từ epoch 75)
- Patience=15 đủ early stop nếu plateau
- 70 epochs trên P100 ≈ **2-2.5 giờ** (P100 nhanh hơn T4)

In [ ]:
model = YOLO('yolo11s.pt')

# Register backup callback
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

results = model.train(
    data=DATA_YAML,
    epochs=70,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    cos_lr=True,
    patience=15,
    cache=True,
    amp=True,
    device=0,
    project='/kaggle/working/runs',
    name='yolov11s_iphone',
    exist_ok=True,
)

print('\nTraining done!')
print(f'Best weights: {results.save_dir}/weights/best.pt')
print(f'Backup: {BACKUP_DIR}/best.pt')

## 7. Evaluate test set

In [ ]:
best_pt = f'{BACKUP_DIR}/best.pt'
model = YOLO(best_pt)

metrics = model.val(data=DATA_YAML, split='test', imgsz=640)
print(f'\n=== TEST SET METRICS ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'\nPer-class mAP@50:')
for i, name in enumerate(cfg['names']):
    print(f'  {name:<20} {metrics.box.maps[i]:.4f}')

## 8. Final save — copy về /kaggle/working/ (auto-persist when commit)

Khi click **Save Version → Save & Run All** ở Kaggle, mọi file trong `/kaggle/working/` được commit vào output dataset của notebook → có thể download bất cứ lúc nào.

In [ ]:
FINAL_OUT = '/kaggle/working/final'
os.makedirs(FINAL_OUT, exist_ok=True)

# Copy best/last
shutil.copy(f'{BACKUP_DIR}/best.pt', f'{FINAL_OUT}/best.pt')
shutil.copy(f'{BACKUP_DIR}/last.pt', f'{FINAL_OUT}/last.pt')

# Copy graphs
src_dir = '/kaggle/working/runs/yolov11s_iphone'
for fname in ['results.png', 'results.csv', 'confusion_matrix.png',
              'confusion_matrix_normalized.png', 'F1_curve.png',
              'PR_curve.png', 'P_curve.png', 'R_curve.png']:
    src = f'{src_dir}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{FINAL_OUT}/{fname}')

print(f'Saved to: {FINAL_OUT}')
!ls -la {FINAL_OUT}
print('\n👉 Click "Save Version" ở Kaggle để commit output')